# Introducción práctica a scikit-learn

Este notebook explica lo **básico pero importante** para trabajar con `scikit-learn`, especialmente:

- qué es `fit`, `transform`, `fit_transform` y `predict`
- qué es un `Pipeline`
- qué es un `ColumnTransformer`
- cómo combinar preprocesamiento y modelo en un flujo limpio
- por qué esto evita errores comunes

La idea es dejar una base clara para que luego puedas usar modelos más complejos sin perderte en la estructura.

## 1. Importar librerías

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

## 2. Crear un dataset sencillo

Vamos a usar el dataset de **iris** y le agregaremos algunas columnas categóricas y valores faltantes para simular un caso más real.

In [2]:
iris = load_iris(as_frame=True)
X = iris.data.copy()
y = iris.target.copy()

X.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [3]:
X['petal_length_level'] = pd.cut(
    X['petal length (cm)'],
    bins=[0, 2, 5, 10],
    labels=['small', 'medium', 'large']
)

X['sepal_width_group'] = np.where(X['sepal width (cm)'] >= X['sepal width (cm)'].median(), 'wide', 'narrow')

X.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),petal_length_level,sepal_width_group
0,5.1,3.5,1.4,0.2,small,wide
1,4.9,3.0,1.4,0.2,small,wide
2,4.7,3.2,1.3,0.2,small,wide
3,4.6,3.1,1.5,0.2,small,wide
4,5.0,3.6,1.4,0.2,small,wide


In [4]:
np.random.seed(52)
missing_idx_1 = np.random.choice(X.index, size=20, replace=False)
missing_idx_2 = np.random.choice(X.index, size=15, replace=False)

X.loc[missing_idx_1, 'sepal length (cm)'] = np.nan
X.loc[missing_idx_2, 'petal_length_level'] = np.nan

X.isna().sum()

sepal length (cm)     20
sepal width (cm)       0
petal length (cm)      0
petal width (cm)       0
petal_length_level    15
sepal_width_group      0
dtype: int64

## 3. Train / test split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=52
)

X_train.shape, X_test.shape

((112, 6), (38, 6))

## 4. Qué hace sklearn en general

Muchos objetos de `scikit-learn` siguen una lógica parecida:

- `fit`: aprende algo a partir de los datos
- `transform`: transforma los datos usando lo que aprendió
- `fit_transform`: hace ambas cosas seguidas
- `predict`: genera predicciones
- `predict_proba`: genera probabilidades, si el modelo lo soporta

No todos los objetos tienen todos estos métodos, pero esta es la lógica general.

## 5. Ejemplo básico con un imputador

In [6]:
imputer = SimpleImputer(strategy='median')

In [7]:
X_train_num = X_train[['sepal length (cm)', 'sepal width (cm)']]
X_test_num = X_test[['sepal length (cm)', 'sepal width (cm)']]

X_train_num.head()

,sepal length (cm),sepal width (cm)
144,6.7,3.3
20,5.4,3.4
128,6.4,2.8
3,NaN,3.1
85,6.0,3.4


In [8]:
imputer.fit(X_train_num)

,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


Con `fit`, el imputador aprende qué valor usar. En este caso aprende la **mediana** de cada columna usando solo el train.

In [9]:
imputer.statistics_

array([5.8, 3. ])

In [10]:
X_train_num_imputed = imputer.transform(X_train_num)
X_test_num_imputed = imputer.transform(X_test_num)

X_train_num_imputed[:5]

array([[6.7, 3.3],
       [5.4, 3.4],
       [6.4, 2.8],
       [5.8, 3.1],
       [6. , 3.4]])

Fíjate en algo importante:

- el imputador se entrena con `X_train`
- luego transforma `X_train` y `X_test`
- **no** se debe hacer `fit` por separado en test

Eso evita leakage o fuga de información.

## 6. Ejemplo básico con un escalador

In [11]:
scaler = StandardScaler()
scaler.fit(X_train_num_imputed)

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [12]:
X_train_scaled = scaler.transform(X_train_num_imputed)
X_test_scaled = scaler.transform(X_test_num_imputed)

X_train_scaled[:5]

array([[ 1.27736487,  0.5513748 ],
       [-0.59182433,  0.7861808 ],
       [ 0.84601352, -0.62265519],
       [-0.01668919,  0.0817628 ],
       [ 0.27087838,  0.7861808 ]])

In [13]:
pd.DataFrame(X_train_scaled, columns=X_train_num.columns).describe().T

,count,mean,std,min,25%,50%,75%,max
sepal length (cm),112.0,1.742654e-15,1.004494,-2.029662,-0.591824,-0.016689,0.702230,2.715203
sepal width (cm),112.0,8.049117e-16,1.004494,-2.031491,-0.622655,-0.153043,0.786181,3.134241


`StandardScaler` centra y escala las variables para que queden aproximadamente con:

- media cercana a 0
- desviación estándar cercana a 1

## 7. Problema del flujo manual

Si haces todo a mano, el flujo se vuelve largo y propenso a errores:

1. separar columnas
2. imputar numéricas
3. escalar numéricas
4. imputar categóricas
5. codificar categóricas
6. unir todo
7. entrenar modelo

Eso funciona, pero se vuelve incómodo cuando el proyecto crece.

Para eso existen `Pipeline` y `ColumnTransformer`.

## 8. Qué es un Pipeline

Un `Pipeline` encadena pasos en orden.

Ejemplo conceptual:

```python
Pipeline([
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])
```

La idea es:

- el primer paso transforma
- el siguiente transforma
- el último normalmente es el modelo

Ventajas:

- código más limpio
- menos errores
- más fácil hacer validación cruzada
- más fácil cambiar componentes

## 9. Primer Pipeline simple solo con variables numéricas

In [14]:
num_cols = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

numeric_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a f

In [15]:
X_train_num_processed = numeric_pipeline.fit_transform(X_train[num_cols])
X_test_num_processed = numeric_pipeline.transform(X_test[num_cols])

X_train_num_processed[:5]

array([[ 1.27736487,  0.5513748 ,  1.08043525,  1.64632619],
       [-0.59182433,  0.7861808 , -1.19416528, -1.34024864],
       [ 0.84601352, -0.62265519,  1.02357024,  1.12692187],
       [-0.01668919,  0.0817628 , -1.30789531, -1.34024864],
       [ 0.27087838,  0.7861808 ,  0.39805509,  0.47766647]])

Acá el pipeline hace internamente:

1. `fit` del imputador
2. `transform` del imputador
3. `fit` del scaler
4. `transform` del scaler

Todo de forma ordenada.

## 10. Qué es un ColumnTransformer

`ColumnTransformer` sirve para aplicar **transformaciones distintas a distintos grupos de columnas**.

Esto es clave porque normalmente:

- las numéricas se imputan y escalan
- las categóricas se imputan y se convierten con one-hot encoding

No tiene sentido tratar todas las columnas igual.

In [16]:
num_cols = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
cat_cols = ['petal_length_level', 'sepal_width_group']

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, num_cols),
    ('cat', categorical_pipeline, cat_cols)
])

preprocess

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [17]:
X_train_processed = preprocess.fit_transform(X_train)
X_test_processed = preprocess.transform(X_test)

X_train_processed[:5]

array([[ 1.27736487,  0.5513748 ,  1.08043525,  1.64632619,  1.        ,
         0.        ,  0.        ,  0.        ,  1.        ],
       [-0.59182433,  0.7861808 , -1.19416528, -1.34024864,  0.        ,
         0.        ,  1.        ,  0.        ,  1.        ],
       [ 0.84601352, -0.62265519,  1.02357024,  1.12692187,  1.        ,
         0.        ,  0.        ,  1.        ,  0.        ],
       [-0.01668919,  0.0817628 , -1.30789531, -1.34024864,  0.        ,
         0.        ,  1.        ,  0.        ,  1.        ],
       [ 0.27087838,  0.7861808 ,  0.39805509,  0.47766647,  0.        ,
         1.        ,  0.        ,  0.        ,  1.        ]])

In [18]:
preprocess.get_feature_names_out()

array(['num__sepal length (cm)', 'num__sepal width (cm)',
       'num__petal length (cm)', 'num__petal width (cm)',
       'cat__petal_length_level_large', 'cat__petal_length_level_medium',
       'cat__petal_length_level_small', 'cat__sepal_width_group_narrow',
       'cat__sepal_width_group_wide'], dtype=object)

Esto ya es mucho más cercano a un caso real:

- unas columnas van por pipeline numérico
- otras van por pipeline categórico
- al final todo se concatena en una sola matriz lista para modelar

## 11. ¿Por qué no hacer dummies con pandas antes?

Sí se puede, pero `ColumnTransformer` tiene ventajas:

- mantiene el flujo dentro de sklearn
- evita inconsistencias entre train y test
- se integra mejor con `Pipeline`, CV y búsqueda de hiperparámetros
- maneja categorías nuevas con `handle_unknown='ignore'`

## 12. Pipeline completo: preprocesamiento + modelo

Ahora vamos a unir todo en un solo objeto:

1. preprocesamiento
2. modelo

In [19]:
model = Pipeline(steps=[
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=2000))
])

model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains spa

In [20]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains spa

In [21]:
y_pred = model.predict(X_test)
y_pred[:10]

array([2, 0, 0, 0, 1, 0, 1, 0, 1, 0])

In [22]:
accuracy_score(y_test, y_pred)

0.9736842105263158

In [23]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        13
           1       0.93      1.00      0.96        13
           2       1.00      0.92      0.96        12

    accuracy                           0.97        38
   macro avg       0.98      0.97      0.97        38
weighted avg       0.98      0.97      0.97        38



## 13. Qué pasa internamente cuando hago model.fit(...)

Cuando haces esto:

```python
model.fit(X_train, y_train)
```

scikit-learn hace algo parecido a esto:

1. aplica `fit_transform` al paso `prep`
2. toma la salida transformada
3. entrena `clf` con esa salida

Y cuando haces:

```python
model.predict(X_test)
```

hace:

1. aplica `transform` al paso `prep`
2. pasa esa salida al clasificador
3. genera predicciones

## 14. Ver el preprocesamiento ya aprendido

In [24]:
model.named_steps['prep']

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [25]:
model.named_steps['clf']

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [26]:
feature_names = model.named_steps['prep'].get_feature_names_out()
feature_names

array(['num__sepal length (cm)', 'num__sepal width (cm)',
       'num__petal length (cm)', 'num__petal width (cm)',
       'cat__petal_length_level_large', 'cat__petal_length_level_medium',
       'cat__petal_length_level_small', 'cat__sepal_width_group_narrow',
       'cat__sepal_width_group_wide'], dtype=object)

## 15. Ver los coeficientes del modelo

In [27]:
coefs = model.named_steps['clf'].coef_
coefs.shape

(3, 9)

In [28]:
coef_df = pd.DataFrame(coefs.T, index=feature_names, columns=[f'class_{i}' for i in range(coefs.shape[0])])
coef_df.head(15)

,class_0,class_1,class_2
num__sepal length (cm),-0.536217,0.390996,0.145222
num__sepal width (cm),0.706242,-0.098403,-0.607839
num__petal length (cm),-1.537109,-0.157896,1.695005
num__petal width (cm),-1.430588,-0.667357,2.097946
cat__petal_length_level_large,-0.049180,-0.276938,0.326118
cat__petal_length_level_medium,-0.530940,0.854873,-0.323933
cat__petal_length_level_small,0.585985,-0.581310,-0.004675
cat__sepal_width_group_narrow,-0.274642,0.128982,0.145660
cat__sepal_width_group_wide,0.280506,-0.132356,-0.148150


## 16. Ventaja clave: todo queda en un solo objeto

Eso significa que puedes:

- guardar el pipeline completo
- usarlo en validación cruzada
- cambiar el modelo sin reescribir todo
- evitar olvidar pasos de preprocesamiento

## 17. Ejemplo de cambio rápido de modelo

In [29]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(steps=[
    ('prep', preprocess),
    ('clf', RandomForestClassifier(n_estimators=200, random_state=52))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

accuracy_score(y_test, rf_pred)

0.9473684210526315

Fíjate que el preprocesamiento se reutiliza. Solo cambió el clasificador.

## 18. Integración con train/test y validación cruzada

Una razón muy importante para usar pipelines es que funcionan muy bien con herramientas de sklearn como:

- `train_test_split`
- `cross_val_score`
- `GridSearchCV`
- `RandomizedSearchCV`

Porque todo el flujo queda encapsulado.

In [30]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=52)

scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
scores

array([0.86956522, 1.        , 0.95454545, 0.90909091, 0.86363636])

In [31]:
print('Accuracy CV promedio:', scores.mean())

Accuracy CV promedio: 0.9193675889328062


Esto es mucho mejor que hacer preprocesamiento manual por fuera y luego pasar datos transformados, porque el pipeline evita leakage dentro de cada fold.

## 19. Error común: hacer fit en test

### Incorrecto

```python
scaler.fit(X_test)
```

### Correcto

```python
scaler.fit(X_train)
scaler.transform(X_test)
```

Cuando usas `Pipeline`, esta separación correcta ocurre automáticamente.

## 20. Error común: olvidar transformar el test

Otro error común es entrenar un modelo con datos escalados o codificados, pero luego predecir sobre un test sin transformar.

Con `Pipeline`, eso también se evita porque el mismo objeto se encarga del flujo completo.

## 21. Resumen mental simple

Puedes pensar así:

### Transformer
Objeto que aprende algo de X y transforma X.

Ejemplos:
- `SimpleImputer`
- `StandardScaler`
- `OneHotEncoder`

### Estimator / model
Objeto que aprende a predecir y.

Ejemplos:
- `LogisticRegression`
- `RandomForestClassifier`

### Pipeline
Cadena ordenada de transformers + modelo.

### ColumnTransformer
Forma de aplicar pipelines distintos a columnas distintas.

## 22. Estructura típica de un proyecto sklearn

In [32]:
num_cols = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
cat_cols = ['petal_length_level', 'sepal_width_group']

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, num_cols),
    ('cat', categorical_pipeline, cat_cols)
])

final_model = Pipeline(steps=[
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=2000))
])

final_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prep', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains spa

## 23. Conclusiones

Ideas clave:

1. `fit` aprende a partir de train.
2. `transform` aplica una transformación ya aprendida.
3. `Pipeline` encadena pasos en orden.
4. `ColumnTransformer` aplica tratamientos distintos por tipo de columna.
5. Juntos, permiten construir flujos limpios, reproducibles y sin leakage.
6. Si aprendes bien esta estructura, luego usar sklearn se vuelve mucho más fácil.

## 24. Ejercicio sugerido

Prueba estos cambios:

- cambia `LogisticRegression` por otro clasificador
- agrega otra columna categórica
- cambia la estrategia del imputador
- usa `cross_val_score` con otra métrica
- intenta inspeccionar `get_feature_names_out()` después del preprocesamiento